In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


In [11]:
text = "hello i am learning Transformers using google colab"

In [12]:
token = tokenizer.tokenize(text)

In [13]:
print(token)

['hello', 'i', 'am', 'learning', 'transformers', 'using', 'google', 'cola', '##b']


In [14]:
token_ids = tokenizer.convert_tokens_to_ids(token)
print(token_ids)

[7592, 1045, 2572, 4083, 19081, 2478, 8224, 15270, 2497]


In [15]:
encode = tokenizer.encode(text,return_tensors='pt')
print(encode)

tensor([[  101,  7592,  1045,  2572,  4083, 19081,  2478,  8224, 15270,  2497,
           102]])


In [16]:
decoder = tokenizer.decode(encode[0])
print(decoder)

[CLS] hello i am learning transformers using google colab [SEP]


In [17]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "AI is transforming the world."

# Full encoding with all components
encoding = tokenizer(
    text,
    return_tensors="pt",       # Return PyTorch tensors
    padding=True,
    truncation=True,
    max_length=20
)

print("=== What goes into a Transformer ===")
print()

# 1. input_ids: the token IDs
print("input_ids:", encoding["input_ids"])
print("Shape:", encoding["input_ids"].shape)  # [batch_size, seq_length]
print()

# 2. attention_mask: 1 = real token, 0 = padding (ignore this)
print("attention_mask:", encoding["attention_mask"])
print()

# 3. token_type_ids: which sentence each token belongs to (for 2-sentence tasks)
print("token_type_ids:", encoding["token_type_ids"])
# 0 = first sentence, 1 = second sentence

=== What goes into a Transformer ===

input_ids: tensor([[  101,  9932,  2003, 17903,  1996,  2088,  1012,   102]])
Shape: torch.Size([1, 8])

attention_mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1]])

token_type_ids: tensor([[0, 0, 0, 0, 0, 0, 0, 0]])


In [18]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

def get_sentence_embedding(text):
    """Get a single vector representing the whole sentence"""
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
    # Mean pooling: average all token embeddings
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings

# Compare sentences
sentence_a = "I love machine learning"
sentence_b = "I enjoy deep learning"
sentence_c = "The weather is sunny today"

emb_a = get_sentence_embedding(sentence_a)
emb_b = get_sentence_embedding(sentence_b)
emb_c = get_sentence_embedding(sentence_c)

# Cosine similarity: 1.0 = identical meaning, 0.0 = completely different
sim_ab = F.cosine_similarity(emb_a, emb_b).item()
sim_ac = F.cosine_similarity(emb_a, emb_c).item()

print(f"'{sentence_a}' vs '{sentence_b}' → Similarity: {sim_ab:.4f}")
print(f"'{sentence_a}' vs '{sentence_c}' → Similarity: {sim_ac:.4f}")
# Expected: sim_ab >> sim_ac  (semantically closer sentences = higher score)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


'I love machine learning' vs 'I enjoy deep learning' → Similarity: 0.7857
'I love machine learning' vs 'The weather is sunny today' → Similarity: 0.5830


In [19]:
from transformers import pipeline

# pipeline = high-level API that wraps everything (tokenizer + model + post-processing)
classifier = pipeline("sentiment-analysis")

sentences = [
    "This lecture is absolutely amazing!",
    "I don't understand this topic at all.",
    "The exam was okay, not too hard.",
]

results = classifier(sentences)

print("=== Sentiment Analysis ===")
for sentence, result in zip(sentences, results):
    label = result["label"]
    score = result["score"]
    emoji = "😊" if label == "POSITIVE" else "😞"
    print(f"{emoji} '{sentence}'")
    print(f"   → {label} (confidence: {score:.2%})\n")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

=== Sentiment Analysis ===
😊 'This lecture is absolutely amazing!'
   → POSITIVE (confidence: 99.99%)

😞 'I don't understand this topic at all.'
   → NEGATIVE (confidence: 99.96%)

😊 'The exam was okay, not too hard.'
   → POSITIVE (confidence: 99.87%)



In [21]:
from transformers import pipeline

fill_mask = pipeline(
    "fill-mask",
    model="bert-base-uncased"
)

result = fill_mask(
    "I love machine learning but deep learning is [MASK] ."
)

for r in result:
    print(r["token_str"])

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


important
hard
difficult
impossible
good


In [22]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="gpt2"
)

result = generator(
    "Artificial Intelligence is",
    max_length=30
)

print(result[0]["generated_text"])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Artificial Intelligence is the next major player in this field. This is a promising field for the future. The AI's ability to detect, analyze, and understand behavior is impressive. I think that's what's required for AI to improve the world. The AI's ability to learn will be important. The AI's ability to learn will be important. It's a challenge to understand the human mind, to understand the human brain, and to understand language. The AI's ability to understand human language will be important. The AI's ability to learn will be important. The AI's ability to learn will be important. I think this will be a good thing for humanity. It will allow us to help people understand the world and to get them to understand human language. It will allow us to help them to understand human language. We can put in our own language, we can write our own language, we can learn our own language, we can learn our own language, we can learn our own language, we can learn our own language. The human min

In [23]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset
import torch

# --- Your training data ---
train_data = {
    "text": [
        "This product is great!",
        "Terrible quality, broke in one day.",
        "Average product, nothing special.",
        "Absolutely love it, highly recommend!",
        "Worst purchase ever, very disappointed.",
    ],
    "label": [1, 0, 1, 1, 0]   # 1 = positive, 0 = negative
}

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2  # binary classification
)

# Tokenize dataset
def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True, max_length=64)

dataset = Dataset.from_dict(train_data)
dataset = dataset.map(tokenize, batched=True)

# Training configuration
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    logging_steps=5,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

# Train!
trainer.train()
print("✅ Fine-tuning complete!")

# Inference
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=64)
    with torch.no_grad():
        logits = model(**inputs).logits
    pred = torch.argmax(logits, dim=1).item()
    return "POSITIVE 😊" if pred == 1 else "NEGATIVE 😞"

print(predict("This is an ex cellent course!"))
print(predict("I hated every moment of this."))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
5,0.655799


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Fine-tuning complete!
POSITIVE 😊
NEGATIVE 😞
